# CSE 150A: Bayesian Network for Programming Skill Prediction

**Team:** Iha Gadiya, Ioanna Gkerdouki, Lianna Lim, Ved Panse, William Diego

**Goal:** Model how AI tool usage influences programming skill (proxied by salary)

## Part 4: Evaluation Functions (Iha)

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

def get_trainTestSplit(dataLink):
    #given this data is filtered
    data = pd.read_csv(dataLink)

    train_data, test_data = train_test_split(
        data,
        test_size=0.3,
        random_state=42
    )

    return train_data, test_data

In [ ]:
#Set up baseline
#Probability of most common yearly comp ranege
def baseLine_calc(data):
    q1 = data['ConvertedCompYearly'].quantile(0.25)
    q2 = data['ConvertedCompYearly'].quantile(0.5)
    q3 = data['ConvertedCompYearly'].quantile(0.75)

    counts = [0, 0, 0, 0]

    for val in data['ConvertedCompYearly']:
        if val < q1:
            counts[0]+=1
        elif val >= q1 and val < q2:
            counts[1]+= 1
        elif val >= q2 and val < q3:
            counts[2]+= 1
        else:
            counts[3]+= 1

    return max(counts)/sum(counts)

In [ ]:
def accuracy_test(predicted_labels, test_data):
    correct = 0
    total = len(predicted_labels)

    true_labels = test_data['ConvertedCompYearly'].values

    for i in range(total):
        if true_labels[i] == predicted_labels[i]:
            correct += 1


    return correct/total

## Part 3: Bayesian Network Structure & Coding (Ioanna)

In [ ]:
# Imports for Bayesian Network
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.inference import VariableElimination
import numpy as np

In [ ]:
# Load the cleaned data
DATA_PATH = '../cleaned_data/cleaned_stackoverflow_bn_data.csv'

data = pd.read_csv(DATA_PATH)

# Drop Country column (used for filtering, not in our DAG)
data = data.drop(columns=['Country'])

print(f"Total rows: {len(data)}")
print(f"Columns: {list(data.columns)}")
print(f"\nSample data:")
data.head()

In [ ]:
# Check unique values for each variable
print("Unique values per column:")
for col in data.columns:
    print(f"  {col}: {data[col].unique()}")

In [ ]:
# Define the DAG structure based on our proposal
# Format: (parent, child) - parent influences child

edges = [
    ('YearsCode', 'YearsCodePro'),         # Total experience -> Professional experience
    ('YearsCode', 'CodingActivities'),     # Total experience -> Coding activities
    ('YearsCode', 'ConvertedCompYearly'),  # Total experience -> Salary
    ('YearsCodePro', 'ConvertedCompYearly'),   # Professional exp -> Salary
    ('AISelect', 'ConvertedCompYearly'),       # AI usage -> Salary
    ('CodingActivities', 'ConvertedCompYearly'),  # Activities -> Salary
    ('OrgSize', 'ConvertedCompYearly'),        # Company size -> Salary
    ('DevType', 'ConvertedCompYearly'),        # Role type -> Salary
]

# Create the Bayesian Network
model = DiscreteBayesianNetwork(edges)

print("Bayesian Network created!")
print(f"Nodes: {list(model.nodes())}")
print(f"Edges: {list(model.edges())}")

In [ ]:
# Split data into train and test sets
train_data, test_data = get_trainTestSplit(DATA_PATH)

# Drop Country from train/test as well
train_data = train_data.drop(columns=['Country'])
test_data = test_data.drop(columns=['Country'])

print(f"Training set: {len(train_data)} rows")
print(f"Test set: {len(test_data)} rows")

In [ ]:
# Fit the model using Maximum Likelihood Estimation (MLE)
# MLE formula: P(X | Parents) = count(X, Parents) / count(Parents)

model.fit(train_data, estimator=DiscreteMLE())

print("Model fitted with MLE!")
print(f"Model is valid: {model.check_model()}")

In [ ]:
# View the Conditional Probability Tables (CPTs)
print("=" * 60)
print("CONDITIONAL PROBABILITY TABLES")
print("=" * 60)

for cpd in model.get_cpds():
    print(f"\n--- CPT for {cpd.variable} ---")
    parents = cpd.get_evidence()
    if parents:
        print(f"Parents: {parents}")
    else:
        print("Parents: None (root node)")
    print(cpd)

## Inference: Query the Model

In [ ]:
# Create inference engine
inference = VariableElimination(model)

In [ ]:
# Query 1: P(Salary | uses AI tools)
print("P(ConvertedCompYearly | AISelect = yes)")
print("="*40)
result = inference.query(['ConvertedCompYearly'], evidence={'AISelect': 'yes'})
print(result)

In [ ]:
# Query 2: P(Salary | doesn't use AI tools)
print("P(ConvertedCompYearly | AISelect = no)")
print("="*40)
result = inference.query(['ConvertedCompYearly'], evidence={'AISelect': 'no'})
print(result)

In [ ]:
# Query 3: P(Salary | high experience AND uses AI)
print("P(ConvertedCompYearly | YearsCodePro=high, AISelect=yes)")
print("="*50)
result = inference.query(
    ['ConvertedCompYearly'],
    evidence={'YearsCodePro': 'high', 'AISelect': 'yes'}
)
print(result)

## AI Usage Effect Analysis

In [ ]:
# Compare salary distributions across AI usage levels
print("=" * 60)
print("AI USAGE EFFECT ON PROGRAMMING SKILL (Salary Proxy)")
print("=" * 60)

ai_states = data['AISelect'].unique()

for ai_state in ai_states:
    print(f"\nP(Salary | AISelect = {ai_state}):")
    result = inference.query(['ConvertedCompYearly'], evidence={'AISelect': ai_state})

    # Extract probabilities
    for i, state in enumerate(result.state_names['ConvertedCompYearly']):
        prob = result.values[i]
        print(f"  {state}: {prob:.3f}")

## Model Evaluation: Predictions on Test Set

In [ ]:
def predict_salary(row, model, inference):
    """
    Predict most likely salary category given other variables.
    Returns None if evidence contains states not seen during training.
    """
    evidence = {
        'YearsCode': row['YearsCode'],
        'YearsCodePro': row['YearsCodePro'],
        'AISelect': row['AISelect'],
        'CodingActivities': row['CodingActivities'],
        'OrgSize': row['OrgSize'],
        'DevType': row['DevType'],
    }

    try:
        result = inference.query(['ConvertedCompYearly'], evidence=evidence)

        # Get the state with highest probability
        max_idx = np.argmax(result.values)
        predicted = result.state_names['ConvertedCompYearly'][max_idx]

        return predicted
    except KeyError:
        # Handle unseen states in test data
        return None

In [ ]:
# Make predictions on test set
print("Making predictions on test set...")

predictions = []
skipped = 0
for idx, row in test_data.iterrows():
    pred = predict_salary(row, model, inference)
    if pred is None:
        skipped += 1
    predictions.append(pred)

print(f"Made {len(predictions) - skipped} predictions")
if skipped > 0:
    print(f"Skipped {skipped} rows with unseen states")

In [ ]:
# Calculate accuracy (excluding rows with unseen states)
actual = test_data['ConvertedCompYearly'].values
valid_pairs = [(p, a) for p, a in zip(predictions, actual) if p is not None]
correct = sum(1 for p, a in valid_pairs if p == a)
accuracy = correct / len(valid_pairs) if valid_pairs else 0

print("=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"\nBayesian Network Accuracy: {accuracy:.2%}")
print(f"Correct predictions: {correct} / {len(valid_pairs)}")

In [ ]:
# Compare to baseline (majority class)
# Since data is already discretized, find most common class
majority_class = train_data['ConvertedCompYearly'].value_counts().idxmax()
baseline_correct = sum(1 for a in actual if a == majority_class)
baseline_accuracy = baseline_correct / len(actual)

print(f"Baseline (majority class '{majority_class}'): {baseline_accuracy:.2%}")
print(f"\nImprovement over baseline: {(accuracy - baseline_accuracy)*100:.1f} percentage points")

## Summary

This Bayesian Network models how various factors influence programming skill (proxied by yearly compensation):

**Key Findings:**
- The model captures conditional dependencies between experience, AI usage, and compensation
- We can query P(Skill | AI usage) to analyze AI tool effects
- Model accuracy compared against majority class baseline